# Chapter 3: Handling Sequences with PyTorch
**Module 02 – Intermediate Deep Learning with PyTorch**

> *Instructor: Michal Oleszak, Machine Learning Engineer*

## 3.1 Sequential Data

Sequential data is **ordered in time or space**, where the order contains meaningful dependencies.

**Examples of sequential data:**
- Time series (stock prices, sensor readings, electricity consumption)
- Text (words depend on previous words)
- Audio waveforms

### Electricity Consumption Prediction Task
Given past electricity consumption readings, predict the next one.
Dataset contains readings from 2011–2015 at 15-minute intervals.

## 3.2 Train-Test Split for Time Series

⚠️ **Never randomly split time series data!**

Random splitting causes **look-ahead bias** — the model sees future data during training.

✅ **Correct approach**: Split by time. Use earlier data for training, later data for testing.

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# Simulate electricity consumption data
np.random.seed(42)
timestamps = pd.date_range('2011-01-01', periods=1000, freq='15T')
consumption = np.sin(np.linspace(0, 50, 1000)) + np.random.normal(0, 0.1, 1000)
df = pd.DataFrame({'timestamp': timestamps, 'consumption': consumption})

# Time-based split (80% train, 20% test)
split_idx = int(len(df) * 0.8)
df_train = df[:split_idx]
df_test = df[split_idx:]

print(f"Train: {df_train.shape}, from {df_train.timestamp.min()} to {df_train.timestamp.max()}")
print(f"Test:  {df_test.shape},  from {df_test.timestamp.min()} to {df_test.timestamp.max()}")

## 3.3 Creating Sequences

In [ ]:
def create_sequences(df, seq_length):
    """Create (input_sequence, target) pairs for time series."""
    xs, ys = [], []
    for i in range(len(df) - seq_length):
        x = df.iloc[i:(i + seq_length), 1].values  # consumption column
        y = df.iloc[i + seq_length, 1]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

# Sequence length = 96 (24 hours × 4 readings/hour)
SEQ_LENGTH = 96

X_train, y_train = create_sequences(df_train, SEQ_LENGTH)
X_test, y_test   = create_sequences(df_test,  SEQ_LENGTH)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"First sequence (first 5 values): {X_train[0, :5]}")
print(f"First target: {y_train[0]:.4f}")

## 3.4 RNN Model for Time Series

In [ ]:
import torch.nn as nn

class ElectricityRNN(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2):
        super(ElectricityRNN, self).__init__()
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True  # input shape: (batch, seq_len, features)
        )
        self.fc = nn.Linear(hidden_size, 1)  # predict one value

    def forward(self, x):
        # x shape: (batch, seq_length, 1)
        out, _ = self.rnn(x)
        # Take output from last timestep
        out = self.fc(out[:, -1, :])
        return out

model = ElectricityRNN(input_size=1, hidden_size=64, num_layers=2)
print(model)

# Test forward pass
dummy = torch.randn(32, SEQ_LENGTH, 1)  # batch=32, seq=96, features=1
print(f"\nOutput shape: {model(dummy).shape}")  # (32, 1)

## 3.5 LSTM — Long Short-Term Memory

LSTMs are better than basic RNNs for long sequences because they have **memory gates** that control what information to keep or forget.

In [ ]:
class ElectricityLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2):
        super(ElectricityLSTM, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, (h_n, c_n) = self.lstm(x)
        out = self.fc(out[:, -1, :])  # Last timestep
        return out

model_lstm = ElectricityLSTM()
print(model_lstm)

## Summary

| Model | Strength | Limitation |
|---|---|---|
| RNN | Simple, fast | Struggles with long-range dependencies |
| LSTM | Handles long sequences | More parameters, slower |
| GRU | Faster than LSTM, fewer params | Slightly less expressive |